# Chest X-ray Multi-Label Classification
### EfficientNetV2-S + Swin-Small Hybrid + TTA
Dataset: **NIH Chest X-ray 14 (224x224 resized)**

**Kaggle setup steps before running this:**
1. Add the dataset: click **+ Add Input** (right sidebar) and search for
   `NIH Chest X-rays (224x224 resized)` (or the original NIH ChestX-ray14
   dataset — either works, this notebook auto-detects the paths).
2. Turn on **GPU** in Settings → Accelerator (T4 x2 or P100).
3. Turn on **Internet** in Settings (needed once, to download ImageNet
   pretrained weights for EfficientNetV2 / Swin via `timm`).
4. Run all cells. The best checkpoint is saved to
   `/kaggle/working/best_model.pth` — download it from the **Output**
   tab when training finishes and drop it into `webapp/weights/` in the
   project locally.


In [ ]:
!pip install -q timm==0.9.16

In [ ]:
import os, sys, glob, time, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 1. Config

In [ ]:
NIH_CLASSES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia",
]
NUM_CLASSES = len(NIH_CLASSES)
IMG_SIZE = 224
BATCH_SIZE = 32          # lower this to 16 if you hit CUDA OOM
EPOCHS = 15
LR = 3e-4
VAL_FRAC = 0.15
SEED = 42


## 2. Locate dataset + build train/val split (split by Patient ID, no leakage)

In [ ]:
def find_dataset_paths():
    csv_candidates = glob.glob("/kaggle/input/**/Data_Entry_2017.csv", recursive=True)
    assert csv_candidates, "Data_Entry_2017.csv not found — did you Add Input the NIH dataset?"
    csv_path = csv_candidates[0]
    dataset_root = os.path.dirname(csv_path)
    img_dir = None
    for cand in ["images", "images_224", "Images", "images-224"]:
        p = os.path.join(dataset_root, cand)
        if os.path.isdir(p):
            img_dir = p
            break
    if img_dir is None:
        for root, dirs, files in os.walk(dataset_root):
            if any(f.lower().endswith((".png", ".jpg", ".jpeg")) for f in files):
                img_dir = root
                break
    assert img_dir, f"Could not find an image folder under {dataset_root}"
    return csv_path, img_dir

csv_path, img_dir = find_dataset_paths()
print("CSV:", csv_path)
print("Images dir:", img_dir)

df = pd.read_csv(csv_path)
print(df.shape)
df.head()


In [ ]:
def build_label_matrix(df):
    labels = np.zeros((len(df), NUM_CLASSES), dtype=np.float32)
    for i, findings in enumerate(df["Finding Labels"]):
        if findings == "No Finding":
            continue
        for f in findings.split("|"):
            f = f.strip()
            if f in NIH_CLASSES:
                labels[i, NIH_CLASSES.index(f)] = 1.0
    return labels

patient_ids = df["Patient ID"].unique()
rng = np.random.RandomState(SEED)
rng.shuffle(patient_ids)
n_val = int(len(patient_ids) * VAL_FRAC)
val_ids = set(patient_ids[:n_val])

val_df = df[df["Patient ID"].isin(val_ids)].reset_index(drop=True)
train_df = df[~df["Patient ID"].isin(val_ids)].reset_index(drop=True)
print("train:", len(train_df), "val:", len(val_df))


## 3. Dataset / DataLoader

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(7),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class NIHChestXrayDataset(Dataset):
    def __init__(self, df, img_dir, transform, labels):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.labels = labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.img_dir, row["Image Index"])
        image = Image.open(path).convert("RGB")
        image = self.transform(image)
        label = self.labels[idx]
        return image, label

train_labels = build_label_matrix(train_df)
val_labels = build_label_matrix(val_df)

train_ds = NIHChestXrayDataset(train_df, img_dir, train_transform, train_labels)
val_ds = NIHChestXrayDataset(val_df, img_dir, val_transform, val_labels)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)


## 4. Model — EfficientNetV2-S + Swin-Small hybrid

In [ ]:
class HybridEffNetSwin(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, pretrained=True, dropout=0.3):
        super().__init__()
        self.effnet = timm.create_model(
            "tf_efficientnetv2_s", pretrained=pretrained, features_only=True, out_indices=(-1,)
        )
        eff_channels = self.effnet.feature_info[-1]["num_chs"]

        self.swin = timm.create_model(
            "swin_small_patch4_window7_224", pretrained=pretrained, num_classes=0, global_pool="avg"
        )
        swin_dim = self.swin.num_features

        self.eff_pool = nn.AdaptiveAvgPool2d(1)
        fusion_dim = eff_channels + swin_dim
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )
        self.last_eff_feature_map = None

    def forward(self, x):
        eff_feat_map = self.effnet(x)[-1]
        self.last_eff_feature_map = eff_feat_map
        eff_vec = self.eff_pool(eff_feat_map).flatten(1)
        swin_vec = self.swin(x)
        fused = torch.cat([eff_vec, swin_vec], dim=1)
        return self.classifier(fused)

model = HybridEffNetSwin(pretrained=True).to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")


## 5. Loss, optimizer, class-imbalance weighting

In [ ]:
# NIH14 is heavily imbalanced (most images are "No Finding" for any given
# class) -> use pos_weight per class so rare diseases aren't ignored.
pos_counts = train_labels.sum(axis=0)
neg_counts = len(train_labels) - pos_counts
pos_weight = torch.tensor(neg_counts / np.clip(pos_counts, 1, None), dtype=torch.float32).to(device)
pos_weight = torch.clamp(pos_weight, max=20.0)  # avoid extreme weights blowing up training

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()


## 6. Train

In [ ]:
from sklearn.metrics import roc_auc_score

def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    aucs = []
    for c in range(NUM_CLASSES):
        if len(np.unique(all_labels[:, c])) > 1:
            aucs.append(roc_auc_score(all_labels[:, c], all_probs[:, c]))
    return np.mean(aucs) if aucs else 0.0

best_auc = 0.0
history = []

for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * imgs.size(0)

    scheduler.step()
    train_loss = running_loss / len(train_ds)
    val_auc = evaluate(model, val_loader)
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_mean_auc": val_auc})
    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} | val_mean_auc={val_auc:.4f} | {time.time()-t0:.0f}s")

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({"model_state_dict": model.state_dict(), "val_mean_auc": val_auc, "classes": NIH_CLASSES},
                   "/kaggle/working/best_model.pth")
        print(f"  -> saved new best (AUC={val_auc:.4f})")

print("Best val mean AUC:", best_auc)


## 7. Download the weights
Open the **Output** tab on the right of the Kaggle notebook editor and
download `best_model.pth`. Place it at `webapp/weights/best_model.pth`
in the project you unzip locally — the Flask app auto-loads it from
there. You will not need to retrain again.

In [ ]:
import pandas as pd
pd.DataFrame(history)